# XGBoost Model Training

Train two XGBoost models for comparison:
- **BALANCED Model**: Trained on balanced data
- **RAW Model**: Trained on raw (unbalanced) data

Hyperparameters are tuned using GridSearchCV (10-fold Stratified CV, optimizing for ROC-AUC Macro).

## 1. Setup

In [1]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_fscore_support
)
import warnings
warnings.filterwarnings('ignore')

# Define paths
BASE_PATH = Path('../../..').resolve()
DATA_PATH = BASE_PATH / 'data' / 'processed' / 'pickle'
MODELS_PATH = BASE_PATH / 'models' / 'xgboost'
RESULTS_PATH = BASE_PATH / 'data' / 'results' / 'xgboost'

MODELS_PATH.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

print(f"✓ Paths configured")

✓ Paths configured


## 2. Load Data

In [2]:
# Load training data (BALANCED and RAW)
with open(DATA_PATH / 'X_train_balanced.pkl', 'rb') as f:
    X_train_balanced = pickle.load(f)
with open(DATA_PATH / 'y_train_balanced.pkl', 'rb') as f:
    y_train_balanced = pickle.load(f)

with open(DATA_PATH / 'X_train_raw.pkl', 'rb') as f:
    X_train_raw = pickle.load(f)
with open(DATA_PATH / 'y_train_raw.pkl', 'rb') as f:
    y_train_raw = pickle.load(f)


# Convert DataFrames to numpy arrays
if isinstance(X_train_balanced, pd.DataFrame):
    X_train_balanced = X_train_balanced.values
if isinstance(X_train_raw, pd.DataFrame):
    X_train_raw = X_train_raw.values

# Ensure targets are Series
if isinstance(y_train_balanced, pd.DataFrame):
    y_train_balanced = y_train_balanced.iloc[:, 0]
if isinstance(y_train_raw, pd.DataFrame):
    y_train_raw = y_train_raw.iloc[:, 0]

print(f"✓ BALANCED Training set: {X_train_balanced.shape}")
print(f"✓ RAW Training set: {X_train_raw.shape}")
print(f"\nClass distribution:")
print(f"  BALANCCED Train: {dict(pd.Series(y_train_balanced).value_counts().sort_index())}")
print(f"  RAW Train: {dict(pd.Series(y_train_raw).value_counts().sort_index())}")

✓ BALANCED Training set: (898, 651)
✓ RAW Training set: (699, 651)

Class distribution:
  BALANCCED Train: {0: np.int64(633), 1: np.int64(265)}
  RAW Train: {0: np.int64(633), 1: np.int64(66)}


## 3. Hyperparameter Tuning with GridSearchCV

In [ ]:

print("\n" + "="*70)
print("HYPERPARAMETER TUNING - GRID SEARCH")
print("="*70)
print("\nConfiguration:")
print("  • Metric: Recall Macro (10-fold Stratified CV)")
print("  • Cross-validation: StratifiedKFold (n_splits=10)")
print("  • Strategy: Tune learning_rate, max_depth, min_child_weight, gamma")

# Define CV strategy
cv_stratified = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define parameter grid
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'min_child_weight': [1, 3, 5],
    'gamma': [0.0, 0.5, 1.0, 2.0],
    'reg_alpha': [0.01, 0.1, 1.0, 5.0, 10.0],      
    'reg_lambda': [0.1, 1.0, 5.0, 10.0],    
}

def create_base_model():
    
    return xgb.XGBClassifier(
        objective='binary:logistic',
        num_class=2,
        n_estimators=150,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=42,
        verbosity=0,
    )

def run_grid_search(X_train, y_train, model_name):
    
    print(f"\n{'-'*70}")
    print(f"Grid Search: {model_name} Model")
    print(f"{'-'*70}")
    
    sample_weights = compute_sample_weight('balanced', y_train)
    
    grid_search = GridSearchCV(
        estimator=create_base_model(),
        param_grid=param_grid,
        cv=cv_stratified,
        scoring='recall_macro',
        n_jobs=-1,
        verbose=0,
    )
    
    print(f"Searching optimal hyperparameters...")
    grid_search.fit(X_train, y_train, sample_weight=sample_weights)
    
    print(f"\n✓ Best hyperparameters ({model_name}):")
    for param, value in grid_search.best_params_.items():
        print(f"    {param}: {value}")
    
    print(f"\n✓ Best Recall Macro (CV): {grid_search.best_score_:.4f}")
    
    # Save CV results
    cv_results = pd.DataFrame(grid_search.cv_results_)
    suffix = 'balanced' if 'BALANCED' in model_name else 'raw'
    results_path = RESULTS_PATH / f'gridsearch_results_{suffix}.csv'
    cv_results.to_csv(results_path, index=False)
    print(f"✓ Grid Search results saved")
    
    return grid_search.best_estimator_, grid_search

# Run grid search for both models
model_balanced, gs_balanced = run_grid_search(X_train_balanced, y_train_balanced, 'BALANCED')
model_raw, gs_raw = run_grid_search(X_train_raw, y_train_raw, 'RAW')

print(f"\n" + "="*70)
print(f"✓ Both models trained successfully")
print(f"="*70)


In [ ]:
'''
from sklearn.utils.class_weight import compute_sample_weight

## 3.5 OPTIONAL: Manual Training (Skip GridSearchCV)

# If you want to skip GridSearchCV and train with manual hyperparameters:

print("\n" + "="*70)
print("MANUAL MODEL TRAINING (WITHOUT GRIDSEARCH)")
print("="*70)
print("\nEdit the hyperparameters below for each model and run this cell")
print("(Comment out the GridSearchCV section above if using this)\n")

# Define hyperparameters for BALANCED model
balanced_params = {
    'learning_rate': 0.05,   # Un aprendizaje lento asegura que no salte conclusiones rápidas
    'max_depth': 5,          # Bajarlo de 6 a 4. 6 es demasiado profundo para ~700-1000 filas
    'min_child_weight': 3,   # Exige que al menos 3 pacientes validen una regla para crearla
    'gamma': 2.0,            # Actívalo. Pone un "precio" mínimo a cada nueva rama del árbol
    'reg_alpha': 0.01,        # L1 (Lasso). Apaga las columnas/features que no aportan nada
    'reg_lambda': 1.0,       # L2 (Ridge). Evita que los pesos de las variables sean exagerados
}

# Define hyperparameters for RAW model
raw_params = {
    'learning_rate': 0.05,   # Lo bajamos de 0.1 para que aprenda más lento pero seguro
    'max_depth': 5,          # Máximo 3. Evita totalmente que cree ramas hiper-especializadas
    'min_child_weight': 3,   # Exigimos más muestras por rama para darla como válida
    'gamma': 2.0,            # Le ponemos una barrera alta para hacer splits.
    'reg_alpha': 0.1,        # Regularización fuerte L1
    'reg_lambda': 1.0,       # Regularización fuerte L2
}

print("BALANCED Model Hyperparameters:")
for param, value in balanced_params.items():
    print(f"  {param}: {value}")

print("\nRAW Model Hyperparameters:")
for param, value in raw_params.items():
    print(f"  {param}: {value}")

# Create model with custom hyperparameters
def create_manual_model(**params):
    """Create XGBoost model with custom hyperparameters."""
    return xgb.XGBClassifier(
        objective='binary:logistic',
        #num_class=2,
        n_estimators=200,
        subsample=0.8,
        colsample_bytree=0.7,
        random_state=42,
        verbosity=0,
        **params
    )

# Train BALANCED model
print(f"\n{'-'*70}")
print(f"Training: BALANCED Model (Manual)")
print(f"{'-'*70}")
sample_weights_balanced = compute_sample_weight('balanced', y_train_balanced)
model_balanced = create_manual_model(**balanced_params)
model_balanced.fit(X_train_balanced, y_train_balanced, sample_weight=sample_weights_balanced)
print(f"✓ BALANCED model trained")

# Train RAW model
print(f"\n{'-'*70}")
print(f"Training: RAW Model (Manual)")
print(f"{'-'*70}")
sample_weights_raw = compute_sample_weight('balanced', y_train_raw)
model_raw = create_manual_model(**raw_params)
model_raw.fit(X_train_raw, y_train_raw, sample_weight=sample_weights_raw)
print(f"✓ RAW model trained")

print(f"\n" + "="*70)
print(f"✓ Both models trained with manual hyperparameters")
print(f"="*70)
'''


MANUAL MODEL TRAINING (WITHOUT GRIDSEARCH)

Edit the hyperparameters below for each model and run this cell
(Comment out the GridSearchCV section above if using this)

BALANCED Model Hyperparameters:
  learning_rate: 0.05
  max_depth: 5
  min_child_weight: 3
  gamma: 2.0
  reg_alpha: 0.01
  reg_lambda: 1.0

RAW Model Hyperparameters:
  learning_rate: 0.05
  max_depth: 5
  min_child_weight: 3
  gamma: 2.0
  reg_alpha: 0.1
  reg_lambda: 1.0

----------------------------------------------------------------------
Training: BALANCED Model (Manual)
----------------------------------------------------------------------
✓ BALANCED model trained

----------------------------------------------------------------------
Training: RAW Model (Manual)
----------------------------------------------------------------------
✓ RAW model trained

✓ Both models trained with manual hyperparameters


## 4. Quick Training Validation

In [5]:
print("\n" + "="*70)
print("TRAINING SET PERFORMANCE")
print("="*70)

# BALANCED Model
y_train_pred_balanced = model_balanced.predict(X_train_balanced)
print(f"\nBALANCED Model:")
print(f"  Accuracy:  {accuracy_score(y_train_balanced, y_train_pred_balanced):.4f}")
print(f"  Precision: {precision_score(y_train_balanced, y_train_pred_balanced, average='macro', zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_train_balanced, y_train_pred_balanced, average='macro', zero_division=0):.4f}")
print(f"  F1-Score:  {f1_score(y_train_balanced, y_train_pred_balanced, average='macro', zero_division=0):.4f}")

# RAW Model
y_train_pred_raw = model_raw.predict(X_train_raw)
print(f"\nRAW Model:")
print(f"  Accuracy:  {accuracy_score(y_train_raw, y_train_pred_raw):.4f}")
print(f"  Precision: {precision_score(y_train_raw, y_train_pred_raw, average='macro', zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_train_raw, y_train_pred_raw, average='macro', zero_division=0):.4f}")
print(f"  F1-Score:  {f1_score(y_train_raw, y_train_pred_raw, average='macro', zero_division=0):.4f}")


TRAINING SET PERFORMANCE

BALANCED Model:
  Accuracy:  0.9744
  Precision: 0.9687
  Recall:    0.9698
  F1-Score:  0.9693

RAW Model:
  Accuracy:  0.9628
  Precision: 0.8587
  Recall:    0.9795
  F1-Score:  0.9072


## 5. Per-Class Metrics (Training Set)

In [6]:
print("\n" + "="*70)
print("PER-CLASS METRICS (TRAINING SET)")
print("="*70)

# BALANCED
prec_s, rec_s, f1_s, sup_s = precision_recall_fscore_support(y_train_balanced, y_train_pred_balanced, average=None)
print(f"\nBALANCED Model:")
df_s = pd.DataFrame({
    'Class': [0, 1],
    'Precision': prec_s,
    'Recall': rec_s,
    'F1-Score': f1_s,
    'Support': sup_s
})
print(df_s.to_string(index=False))

# RAW
prec_r, rec_r, f1_r, sup_r = precision_recall_fscore_support(y_train_raw, y_train_pred_raw, average=None)
print(f"\nRAW Model:")
df_r = pd.DataFrame({
    'Class': [0, 1],
    'Precision': prec_r,
    'Recall': rec_r,
    'F1-Score': f1_r,
    'Support': sup_r
})
print(df_r.to_string(index=False))


PER-CLASS METRICS (TRAINING SET)

BALANCED Model:
 Class  Precision   Recall  F1-Score  Support
     0   0.982595 0.981043  0.981818      633
     1   0.954887 0.958491  0.956685      265

RAW Model:
 Class  Precision   Recall  F1-Score  Support
     0   1.000000 0.958926  0.979032      633
     1   0.717391 1.000000  0.835443       66


## 6. Save Models

In [7]:
# Save BALANCED model
model_path_balanced = MODELS_PATH / 'modelo_xgboost_balanced.pkl'
with open(model_path_balanced, 'wb') as f:
    pickle.dump(model_balanced, f)
print(f"✓ BALANCED model saved: {model_path_balanced}")

# Save RAW model
model_path_raw = MODELS_PATH / 'modelo_xgboost_raw.pkl'
with open(model_path_raw, 'wb') as f:
    pickle.dump(model_raw, f)
print(f"✓ RAW model saved: {model_path_raw}")

print(f"\n✅ Both models ready for evaluation")

✓ BALANCED model saved: /home/pablo/Desktop/Estressss/pdg/models/xgboost/modelo_xgboost_balanced.pkl
✓ RAW model saved: /home/pablo/Desktop/Estressss/pdg/models/xgboost/modelo_xgboost_raw.pkl

✅ Both models ready for evaluation
